# P22 — paired RNA+ATAC workflow and evidence

**Research outcome: inconclusive. This notebook does not claim a new biological result.**

This is one entry point for the current workflow: software tests, source preflight, saved RNA findings, paired-input requirements, offline HEAD examples, reviewed code, and next steps. It uses the repository's existing implementations instead of maintaining a second copy.

## Run safely

Use this notebook inside the accepted P22 checkout with its existing Python 3.11 environment. Select that kernel, then **Restart Kernel and Run All**. No installation cell is included. A downloaded notebook alone needs the accompanying checkout to run.

The first working cell runs the offline parser/capture/preflight tests. Later cells stop if those tests fail. Temporary test fixtures are allowed. No cell downloads datasets, invokes a live transport, starts Docker, performs real model training, changes approvals, or overwrites the old RNA notebook. Existing real-analysis environment flags do not enable those actions here.

Saved RNA findings are displayed, **not recomputed**. Synthetic header examples are **not observations from GEO**. Local Jupyter kernel traffic is not a dataset request.

The complete historical RNA workflow remains in [the original notebook](../P22_down_syndrome_all_in_one.ipynb). Its live-data modes are separate from this offline notebook.

In [ ]:
import hashlib
import json
import os
import platform
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path
from uuid import uuid4

from IPython.display import Code, Markdown, display

assert sys.version_info[:2] == (3, 11), "Select the existing P22 Python 3.11 kernel."
sys.dont_write_bytecode = True
cwd = Path.cwd().resolve()
REPO = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "scripts/launcher_head_capture.py").is_file()
     and (path / "pyproject.toml").is_file()),
    None,
)
assert REPO is not None, "Run this notebook from the accepted P22 checkout."
OBSERVED_AT = datetime.now(UTC).isoformat()
print(json.dumps({
    "python": platform.python_version(), "executable": sys.executable,
    "repository": str(REPO), "observed_at": OBSERVED_AT,
    "mode": "OFFLINE_EVIDENCE_AND_TESTS", "live_enabled": False,
}, indent=2))

In [ ]:
# Test first. This list excludes this notebook's own kernel test to avoid recursion.
test_command = [
    sys.executable, "-m", "pytest", "-q",
    "tests/test_development_head.py",
    "tests/test_development_head_capture.py",
    "tests/test_launcher_head_capture.py",
]
test_environment = dict(os.environ, PYTHONPATH=str(REPO / "src"),
                        PYTHONDONTWRITEBYTECODE="1", PYTEST_DISABLE_PLUGIN_AUTOLOAD="1")
test_run = subprocess.run(
    test_command, cwd=REPO, env=test_environment,
    capture_output=True, text=True, timeout=120, check=False,
)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
OFFLINE_TESTS_PASSED = test_run.returncode == 0
assert OFFLINE_TESTS_PASSED, "Offline tests failed. Stop; do not proceed to later cells."

## 1. Observe the current local inputs

The preflight examines four pinned files, file/path safety, the output destination, and disk headroom. It does not reserve that destination or prove runtime controls. Insufficient disk or changed evidence is a refusal, not a reason to change the limits.

In [ ]:
assert OFFLINE_TESTS_PASSED
sys.path.insert(0, str(REPO / "scripts"))
import launcher_head_capture

assert Path(launcher_head_capture.__file__).resolve() == REPO / "scripts/launcher_head_capture.py"
# Do not create this path. Its absence is part of the preflight contract.
output_candidate = REPO / "reports/generated" / ("notebook-observation-" + uuid4().hex)
preflight_result = launcher_head_capture.preflight(
    repo=REPO, output_dir=output_candidate, observed_at=OBSERVED_AT,
)
print(json.dumps(preflight_result, indent=2))
assert preflight_result["status"] == "OFFLINE_PREFLIGHT_OK", "Preflight refused. Stop here."
assert preflight_result["live_allowed"] is False
assert preflight_result["runtime_controls"] == "UNVERIFIED"
assert preflight_result["output_reserved"] is False
assert not output_candidate.exists()

## 2. The selected source and unchanged resource limits

The candidate URL is a stored proposal, not a verified download link from this run. The object contents, decoded size, full-load memory, and applicable assay classes remain unknown. The limits for a proposed future HEAD do not authorize an object download.

In [ ]:
contract = json.loads((REPO / "configs/development_object_source_contract.json").read_text())
assert contract["status"] == "SOURCE_UNRESOLVED"
assert contract["object_inspection_authorized"] is False
assert contract["current_object_permissions"] == {
    "network_bytes": 0, "decoded_bytes": 0, "output_bytes": 0,
    "execution_seconds": 0, "training_allowed": False,
}
print(json.dumps({
    "status": contract["status"],
    "candidate_payload_url": contract["candidate_payload_url"],
    "candidate_url_basis": contract["candidate_url_basis"],
    "exact_payload_bytes": contract["exact_payload_bytes"],
    "expected_decoded_bytes": contract["expected_decoded_bytes"],
    "expected_peak_memory_bytes": contract["expected_peak_memory_bytes"],
    "current_object_permissions": contract["current_object_permissions"],
    "reader_proof": contract["reader_proof"],
    "future_HEAD_proposal_only": contract["single_next_action_proposal"],
}, indent=2))

## 3. Completed RNA evidence — saved results, not a new run

These two sections reproduce the outcome sections from the unchanged, hash-checked research reports. Software test counts are not biological sample sizes. Reproducibility does not turn an inconclusive result into a positive result.

In [ ]:
record_pins = {
    "docs/EXTERNAL_RNA_REPLICATION_RESULTS_2026-09-08.md":
        "81bb3c313749f4b2751e6259a0c52544d7b10762baddc80597abd36521b96354",
    "docs/RNA_DONOR_INFLUENCE_RESULTS_2026-09-10.md":
        "a8892ffd454898bcdc5594c79e09d55d07d70398be5ce3f8f9b58cde563edd36",
}
saved_records = {}
for relative, expected in record_pins.items():
    with (REPO / relative).open("rb") as handle:
        raw = handle.read(65537)
    assert len(raw) <= 65536, "Saved record exceeds the notebook read limit."
    assert hashlib.sha256(raw).hexdigest() == expected, "Saved RNA evidence changed."
    saved_records[relative] = raw.decode("utf-8")
    print(relative, expected)

rna_report = saved_records["docs/EXTERNAL_RNA_REPLICATION_RESULTS_2026-09-08.md"]
influence_report = saved_records["docs/RNA_DONOR_INFLUENCE_RESULTS_2026-09-10.md"]
assert "## Final evidence package" in rna_report
assert "## Execution and verification" in influence_report
display(Markdown(rna_report.split("## Final evidence package")[0]))
display(Markdown(influence_report.split("## Execution and verification")[0]))

## 4. Paired-input and training status

The raw-library pilot preserved paired counts for its tested donor/library slice. That is not an accepted development cohort. The two tested library peak lists had zero exact common regions; that result does not prove that every possible processed object is unusable.

External release/QC semantics and compatible measured ATAC counts remain unresolved. Known study provenance is not a complete specimen-independence audit. Synthetic training, final refits, and one-shot scoring already exercised the software; no real external multimodal evaluation followed.

The authoritative criteria below come from the unchanged object contract. Each criterion must apply to the actual selected cohorts. Missing regions cannot become fabricated zero counts.

In [ ]:
display(Markdown("\n".join(
    "- " + requirement
    for requirement in contract["scientific_acceptance_requirements_unchanged"]
)))
print("Real paired training allowed:", contract["current_object_permissions"]["training_allowed"])

## 5. Offline examples of the accepted HEAD code

The following header is an invented fixture from `example.invalid`. Its length of 123 bytes has no connection to the real candidate payload. The parser's internal `SOURCE_METADATA_OBSERVED` label describes that fixture only.

The capture-core call supplies **no transport**. It must refuse before any request. Notebook success cannot enable a live connection.

In [ ]:
from capture_development_head import PINNED_URL, capture_head
from validate_development_head import parse_head_evidence

synthetic_header = b"HTTP/1.1 200 OK\r\nContent-Length: 123\r\n\r\n"
demo_metadata = parse_head_evidence(
    synthetic_header, url="https://example.invalid/synthetic.rda.gz",
    observed_at="SYNTHETIC_FIXTURE",
)
assert demo_metadata["declared_payload_bytes"] == 123
assert demo_metadata["publisher_payload_checksum"] is None
print(json.dumps({"evidence_kind": "SYNTHETIC_DEMO_NOT_GEO_OBSERVATION",
                  "parser_result": demo_metadata}, indent=2))

capture_refusal = capture_head(url=PINNED_URL, observed_at=OBSERVED_AT)
assert capture_refusal["capture_status"] == "REFUSED"
assert capture_refusal["reason"] == "LIVE_TRANSPORT_NOT_ENABLED"
assert capture_refusal["requests_attempted"] == 0
assert capture_refusal["body_bytes"] == 0
print(json.dumps(capture_refusal, indent=2))

bad_label = launcher_head_capture.preflight(
    repo=REPO, output_dir=output_candidate, observed_at="bad\nlabel",
)
assert bad_label["reason"] == "BAD_LABEL"
assert bad_label["source_hashes"] == {}
assert bad_label["disk_free_bytes"] is None
print("Invalid-label refusal:", bad_label["reason"])

## 6. Exact implementation used by this notebook

The three current scripts are displayed below. They remain the source of truth; notebook cells import them. Open the collapsed output to inspect the code. The preflight has no live launcher or CLI.

In [ ]:
for relative in (
    "scripts/launcher_head_capture.py",
    "scripts/capture_development_head.py",
    "scripts/validate_development_head.py",
):
    display(Markdown("### " + relative))
    display(Code(filename=str(REPO / relative), language="python"))

## 7. Next steps and stop conditions

| Step | Required evidence | What success permits |
| --- | --- | --- |
| E2-M1 | Reviewed native runtime/control design and bounded offline probe allocations | Implement the live binding |
| E2-M2 / M3 | Real enforcement of memory/no-swap, one total deadline, output limits, safe cleanup, and trusted TLS/input use | Independent controls review |
| E2-M4 | Accepted controls and the existing exact HEAD scope approval | One bounded metadata request only |
| E2-R / E3 | Applicable R workspace/assay reader proof, resource contract, and authorized inspection | Assess actual paired inputs |
| M1–M5 | Accepted release/QC, donor support, measured common regions, provenance, and frozen protocol | Consider the real internal training run |
| M6–M8 | Accepted development results, frozen model, and accepted external cohort | One-shot external evaluation |

Stop on missing evidence, changed hashes, failed controls, or unavailable resources. A successful HEAD does not prove valid RNA/ATAC assays. Do not weaken scientific gates to obtain a positive result.

### Existing implementation and detailed records

- [Full plan](../tasks/plan.md) and [ordered checklist](../tasks/todo.md)
- [Paired-data audit and current execution record](../docs/PAIRED_MULTIOME_AUDIT.md)
- [Dataset roles and experimental question](../docs/PAIRED_DS_MULTIOME_DATASET_OPTIONS.md)
- [RNA outcome and original executed-notebook links](../docs/EXTERNAL_RNA_REPLICATION_RESULTS_2026-09-08.md)
- [RNA donor-influence results](../docs/RNA_DONOR_INFLUENCE_RESULTS_2026-09-10.md)
- [Paired training guide](../docs/PAIRED_MULTIOME_TRAINING.md)
- [Bounded input audit](../scripts/audit_multiome.py), [paired loader](../src/p22/data/multiome.py), [synthetic benchmark](../scripts/train_multiome.py)
- [Fold training](../src/p22/eval/multiome_runner.py) and [frozen refit/scoring](../src/p22/eval/multiome_final.py)

These links expose existing work without silently running its acquisition or training commands.

In [ ]:
notebook_decision = {
    "mode": "OFFLINE_EVIDENCE_AND_TESTS",
    "offline_tests_passed": OFFLINE_TESTS_PASSED,
    "source_status": contract["status"],
    "scientific_outcome": "INCONCLUSIVE",
    "live_requests_attempted": capture_refusal["requests_attempted"],
    "training_performed": False,
    "live_allowed": preflight_result["live_allowed"],
    "output_reserved": preflight_result["output_reserved"],
    "demo_header_is_synthetic": True,
    "next_action": "Review E2-M1 native controls and bounded offline probe allocations.",
}
assert not output_candidate.exists()
print(json.dumps(notebook_decision, indent=2))